In [1]:
import pandas as pd
import numpy as np

rais = spark.sql(  # filtro Osasco
    """
    SELECT * 
    FROM lh_cidade_inteligente_osasco.raw_rais_estab_sp 
    WHERE id_municipio IN ('3534401', '353440')
    """
).toPandas()

cnae1 = pd.read_csv(
    "/lakehouse/default/Files/aux_tables/br_bd_diretorios_brasil_cnae_1.csv",
    dtype={"cnae_1": str},
)
cnae2 = pd.read_csv(
    "/lakehouse/default/Files/aux_tables/br_bd_diretorios_brasil_cnae_2.csv",
    dtype={"subclasse": str},
)


rais["cnae_2_subclasse"] = rais["cnae_2_subclasse"].astype(str)

cnae1 = cnae1[["cnae_1", "descricao_secao"]].rename(
    columns={"descricao_secao": "descricao_secao_cnae_1"}
)

cnae2 = cnae2[["subclasse", "descricao_secao"]].rename(
    columns={
        "subclasse": "cnae_2_subclasse",
        "descricao_secao": "descricao_secao_cnae_2",
    }
)


rais_cnae = rais.merge(cnae1, on="cnae_1", how="left").merge(
    cnae2, on="cnae_2_subclasse", how="left"
)

rais_cnae["descricao_secao_cnae"] = np.where(
    rais_cnae["descricao_secao_cnae_2"].isnull(),
    rais_cnae["descricao_secao_cnae_1"],
    rais_cnae["descricao_secao_cnae_2"],
)

rais_cnae["descricao_secao_cnae"] = rais_cnae["descricao_secao_cnae"].str.capitalize()

rais_anual = rais_cnae.groupby(["ano", "descricao_secao_cnae"], as_index=False).agg(
    {"quantidade_vinculos_ativos": "sum"}
)


# Gold tamanho estabelecimento rais
tamanho_dict = {
    "1": "ZERO",
    "2": "ATE 4",
    "3": "DE 5 A 9",
    "4": "DE 10 A 19",
    "5": "DE 20 A 49",
    "6": "DE 50 A 99",
    "7": "DE 100 A 249",
    "8": "DE 250 A 499",
    "9": "DE 500 A 999",
    "10": "1000 OU MAIS",
    "-1": "IGNORADO",
}

rais_cnae["tamanho_estabelecimento"] = (
    rais_cnae["tamanho_estabelecimento"].map(tamanho_dict).str.capitalize()
)
rais_tamanho_estabelecimento = rais_cnae.groupby(
    ["ano", "tamanho_estabelecimento", "descricao_secao_cnae"], as_index=False
).size()

StatementMeta(, bd2810db-6072-4505-b195-47d1c9a1a9db, 3, Finished, Available, Finished)

IsADirectoryError: [Errno 21] Is a directory: '/lakehouse/default/Files/rais_ftp/rais_anual.csv'

In [7]:
rais_anual.loc[rais_anual['ano'] == rais_anual['ano'].max()]

StatementMeta(, bd2810db-6072-4505-b195-47d1c9a1a9db, 9, Finished, Available, Finished)

,ano,descricao_secao_cnae,quantidade_vinculos_ativos
372,2024,"Administração pública, defesa e seguridade social",18805
373,2024,Alojamento e alimentação,7574
374,2024,"Artes, cultura, esporte e recreação",735
375,2024,Atividades administrativas e serviços compleme...,22038
376,2024,"Atividades financeiras, de seguros e serviços ...",20502
377,2024,Atividades imobiliárias,580
378,2024,"Atividades profissionais, científicas e técnicas",11129
379,2024,Comércio; reparação de veículos automotores e ...,49556
380,2024,Construção,7852
381,2024,Educação,12799


In [3]:
# export
rais_tamanho_estabelecimento.to_csv(
    "/lakehouse/default/Files/gold_rais/gold_rais_tamanho_estabelecimento.csv",
    sep=";",
    index=False,
)

rais_anual.to_csv(
    "/lakehouse/default/Files/gold_rais/rais_anual.csv",
    sep=";",
    index=False
)

StatementMeta(, bd2810db-6072-4505-b195-47d1c9a1a9db, 5, Finished, Available, Finished)